# 10 — Vector Store (`core/vector_store.py`)
Two modes:
- **Mock** (`ENABLE_MOCK=true` or no `OPENAI_API_KEY`) → `_NullVectorStore` — in-memory keyword-scored docs, always returns 6 governance docs
- **Prod** → LangChain `PGVector` backed by PostgreSQL + pgvector extension

Public API:
- `get_vector_store(config)` → store object
- `similarity_search(store, query, k=5)` → `[(Document, score), ...]`

Relevance score range: **0.0 – 1.0**. `knowledge_agent` filters: `score > 0.70`


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'codebase', 'codebase', 'src'))
os.makedirs("logs", exist_ok=True)
os.environ["ENABLE_MOCK"] = "true"  # use NullVectorStore

## 1. get_vector_store — Returns NullVectorStore in Mock Mode

In [ ]:
from core.vector_store import get_vector_store
from config.settings import VectorDBConfig

store = get_vector_store(VectorDBConfig())
print("Store type:", type(store).__name__)
print("Is NullVectorStore:", type(store).__name__ == "_NullVectorStore")

## 2. similarity_search — Basic Query

In [ ]:
from core.vector_store import similarity_search

results = similarity_search(store, "What is gross retention rate?", k=5)
print(f"Returned {len(results)} results:\n")
for doc, score in results:
    print(f"  Score: {score:.2f} | Product: {doc.metadata.get('product')} | Topic: {doc.metadata.get('topic')}")
    print(f"  Content: {doc.page_content[:100]}...")
    print()

## 3. Score Filtering (as done in knowledge_agent)

In [ ]:
# knowledge_agent only keeps docs with score >= 0.70
results = similarity_search(store, "customer acquisition cost payback", k=6)
print("All results:")
for doc, score in results:
    keep = "✅ KEEP" if score >= 0.70 else "❌ FILTER"
    print(f"  {keep} | score={score:.2f} | product={doc.metadata.get('product')}")

relevant = [(d, s) for d, s in results if s >= 0.70]
print(f"\nKept: {len(relevant)} / {len(results)}")

## 4. Keyword Relevance Scoring (NullVectorStore internals)

In [ ]:
# NullVectorStore gives higher scores to docs that contain query words
queries = [
    "retention churn grr nrr",
    "bookings revenue arr",
    "cac acquisition cost payback",
    "ltv lifetime value",
    "data quality rules",
]
for q in queries:
    results = similarity_search(store, q, k=1)
    best_doc, best_score = results[0]
    print(f"Query: '{q[:40]}'")
    print(f"  Best: {best_score:.2f} | {best_doc.metadata.get('product')}/{best_doc.metadata.get('topic')}")
    print()

## 5. Document Metadata Structure

In [ ]:
results = similarity_search(store, "retention metrics", k=6)
print("All document metadata:")
for doc, score in results:
    print(f"  {score:.2f} | {doc.metadata}")

## 6. Mock Docs Reference

In [ ]:
from core.vector_store import _NullVectorStore
mock = _NullVectorStore()
print(f"NullVectorStore has {len(mock._MOCK_DOCS)} pre-loaded docs:\n")
for i, d in enumerate(mock._MOCK_DOCS):
    print(f"  [{i+1}] {d['metadata']['product']}/{d['metadata']['topic']}")
    print(f"       {d['content'][:80]}...")